# Activity 2: Build a Two-Layer HBNN

The two hyperbolic layers are imported explicitly and assembled exactly as ordinary PyTorch layers.

In [1]:
!test -d /content/mlss_hbnn || git clone -q https://github.com/GitZH-Chen/HBNN.git /content/mlss_hbnn
!git -C /content/mlss_hbnn checkout -q d5c79c8eed36a0b7c2f15e9a8fbcd0318216e5b0

In [2]:
import sys
import torch
import torch.nn as nn

sys.path.insert(0, "/content/mlss_hbnn")

from lib.bnn.BFC import BFC
from lib.bnn.BMLR import BMLR
from lib.bnn.Geometry import Stereographic

torch.manual_seed(7)

/content/mlss_hbnn/lib/bnn/Geometry/constantcurvature/hyperboloid.py:11: SyntaxWarning: invalid escape sequence '\m'
  \mathbb{H}_K^n= \left\{x \in \mathbb{R}^{n+1} | \|x\|_{\mathcal{L}}^2=\frac{1}{K}, x_1>0 \right\},
/content/mlss_hbnn/lib/bnn/Geometry/constantcurvature/hyperboloid.py:120: SyntaxWarning: invalid escape sequence '\s'
  """-x_0^2+\sum_{i=1}^n x_i^2=\frac{1}{K} \quad \Rightarrow \quad x_0=\sqrt{\frac{1}{-K}+\sum_{i=1}^n x_i^2}"""
/content/mlss_hbnn/lib/bnn/Geometry/constantcurvature/hyperboloid.py:143: SyntaxWarning: invalid escape sequence '\L'
  """<v, \Lzero> _L """
/content/mlss_hbnn/lib/bnn/Geometry/constantcurvature/frechetmean/backward/ball_backward.py:61: SyntaxWarning: invalid escape sequence '\p'
  \partial T/ \partial y


## Define the network

**Predict:** What shape should the output logits have for a batch of 8 samples and 3 classes?

In [3]:
class TwoLayerHBNN(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=8, n_classes=3, curvature=-1.0):
        super().__init__()
        self.manifold = Stereographic(K=curvature)
        self.hidden = BFC(
            in_dim=input_dim,
            out_dim=hidden_dim,
            metric="poincare",
            K=curvature,
            act="relu",
        )
        self.classifier = BMLR(
            n_classes=n_classes,
            dim=hidden_dim,
            metric="poincare",
            K=curvature,
        )

    def forward(self, x):
        x_hyperbolic = self.manifold.exp0(0.25 * x)
        hidden_hyperbolic = self.hidden(x_hyperbolic)
        logits = self.classifier(hidden_hyperbolic)
        return logits


## Instantiate and run a forward pass

In [4]:
model = TwoLayerHBNN(input_dim=2, hidden_dim=8, n_classes=3)
x = torch.randn(8, 2)
logits = model(x)

print(model)
print("Input shape:", tuple(x.shape))
print("Logit shape:", tuple(logits.shape))
print("Finite logits:", torch.isfinite(logits).all().item())

TwoLayerHBNN(
  (manifold): Stereographic()
  (hidden): BFC(training=True, in_dim=2, out_dim=8, is_bias=True, metric=poincare, K=-1.0000, dropout=0, gyrobias=True, act=<function relu at 0x7a8db54df6a0>, busemann_linear=<function _poincare_busemann_linear at 0x7a8db3d45da0>, manifold=Stereographic())
  (classifier): BMLR(training=True, C=3, d=8, metric=poincare, use_bias=True, K=-1.0000, busemann_logits=<torch.jit.torch.jit.ScriptFunction object at 0x7a8daa8f69f0>)
)
Input shape: (8, 2)
Logit shape: (8, 3)
Finite logits: True


## Inspect the geometric pipeline

In [5]:
x_hyperbolic = model.manifold.exp0(0.25 * x)
hidden_hyperbolic = model.hidden(x_hyperbolic)

print("Euclidean input:", tuple(x.shape))
print("Poincaré input:", tuple(x_hyperbolic.shape))
print("BFC output:", tuple(hidden_hyperbolic.shape))
print("BMLR output:", tuple(model.classifier(hidden_hyperbolic).shape))
print("Largest Poincaré norm:", hidden_hyperbolic.norm(dim=-1).max().item())

Euclidean input: (8, 2)
Poincaré input: (8, 2)
BFC output: (8, 8)
BMLR output: (8, 3)
Largest Poincaré norm: 0.12725181877613068


**Modify:** Change `hidden_dim=8` to `hidden_dim=2` or `hidden_dim=16`. Which parameter shapes change?